In [2]:
import torch
import torch.nn as nn

torch.manual_seed(42)


# =========================================================
# 1. MANUAL LSTM CLASS
# =========================================================

class ManualLSTM(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()

        self.input_size = input_size
        self.hidden_size = hidden_size

        # LSTM has 4 gates:
        # Input, Forget, Candidate, Output

        self.W_ih = nn.Parameter(
            torch.randn(4 * hidden_size, input_size)
        )

        self.W_hh = nn.Parameter(
            torch.randn(4 * hidden_size, hidden_size)
        )

        self.b_ih = nn.Parameter(
            torch.randn(4 * hidden_size)
        )

        self.b_hh = nn.Parameter(
            torch.randn(4 * hidden_size)
        )

    def forward(self, x, h0=None, c0=None):

        # x shape:
        # (sequence_length, batch_size, input_size)

        seq_len, batch_size, _ = x.shape

        # Initial hidden state
        if h0 is None:
            h = torch.zeros(
                batch_size,
                self.hidden_size
            )
        else:
            h = h0

        # Initial cell state
        if c0 is None:
            c = torch.zeros(
                batch_size,
                self.hidden_size
            )
        else:
            c = c0

        outputs = []

        # Process each time step
        for t in range(seq_len):

            x_t = x[t]

            # Calculate all four gates together
            gates = (
                torch.mm(x_t, self.W_ih.T)
                + torch.mm(h, self.W_hh.T)
                + self.b_ih
                + self.b_hh
            )

            # Split into 4 parts
            i, f, g, o = gates.chunk(4, dim=1)

            # Apply activation functions

            # Input gate
            i = torch.sigmoid(i)

            # Forget gate
            f = torch.sigmoid(f)

            # Candidate cell state
            g = torch.tanh(g)

            # Output gate
            o = torch.sigmoid(o)

            # Update cell state
            c = f * c + i * g

            # Update hidden state
            h = o * torch.tanh(c)

            # Store hidden state
            outputs.append(h)

        # Convert list to tensor
        outputs = torch.stack(outputs, dim=0)

        return outputs, h, c

In [3]:
# =========================================================
# 2. MANUAL BIDIRECTIONAL LSTM CLASS
# =========================================================

class ManualBiLSTM(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()

        # Forward LSTM
        self.forward_lstm = ManualLSTM(
            input_size,
            hidden_size
        )

        # Backward LSTM
        self.backward_lstm = ManualLSTM(
            input_size,
            hidden_size
        )
    def forward(self, x):

        # ---------------------------------------------
        # Forward direction
        # ---------------------------------------------

        forward_output, _, _ = self.forward_lstm(x)

        # ---------------------------------------------
        # Backward direction
        # ---------------------------------------------

        # Reverse sequence
        x_reverse = torch.flip(x, dims=[0])

        # Run LSTM on reversed sequence
        backward_output, _, _ = self.backward_lstm(
            x_reverse
        )

        # Restore original time order
        backward_output = torch.flip(
            backward_output,
            dims=[0]
        )

        # ---------------------------------------------
        # Concatenate forward and backward outputs
        # ---------------------------------------------

        output = torch.cat(
            [forward_output, backward_output],
            dim=2
        )

        return output

In [4]:
# =========================================================
# 3. CREATE INPUT DATA
# =========================================================

seq_len = 5
batch_size = 2
input_size = 3
hidden_size = 4

x = torch.randn(
    seq_len,
    batch_size,
    input_size
)

print("Input Shape:")
print(x.shape)


# =========================================================
# 4. CREATE MANUAL BiLSTM
# =========================================================

manual_model = ManualBiLSTM(
    input_size,
    hidden_size
)

manual_output = manual_model(x)

print("\nManual BiLSTM Output Shape:")
print(manual_output.shape)

Input Shape:
torch.Size([5, 2, 3])

Manual BiLSTM Output Shape:
torch.Size([5, 2, 8])


In [5]:
# =========================================================
# 5. CREATE PYTORCH BUILT-IN BiLSTM
# =========================================================

builtin_model = nn.LSTM(
    input_size=input_size,
    hidden_size=hidden_size,
    bidirectional=True
)

builtin_output, _ = builtin_model(x)

print("\nBuilt-in BiLSTM Output Shape:")
print(builtin_output.shape)


# =========================================================
# 6. DISPLAY OUTPUTS
# =========================================================

print("\nManual BiLSTM Output:")
print(manual_output)

print("\nBuilt-in BiLSTM Output:")
print(builtin_output)



Built-in BiLSTM Output Shape:
torch.Size([5, 2, 8])

Manual BiLSTM Output:
tensor([[[-3.0155e-04,  2.0926e-01, -2.7552e-02,  1.8688e-03,  1.5902e-02,
          -6.4514e-02, -2.2969e-03,  4.6227e-02],
         [ 5.2608e-01, -1.2950e-02, -1.6890e-01, -1.6190e-01,  4.9538e-02,
          -1.6731e-01,  1.9674e-01,  1.6700e-01]],

        [[ 2.8156e-01, -3.6816e-02, -4.7180e-02,  1.9878e-01,  3.4874e-01,
          -1.9756e-02, -4.9023e-01, -1.2417e-01],
         [ 3.4773e-03, -3.1558e-01, -4.8122e-01,  4.2195e-02,  2.7644e-01,
          -4.4624e-02,  7.6632e-01,  7.0789e-02]],

        [[ 2.5385e-01, -2.7978e-02, -5.1747e-01, -5.2619e-01,  3.7104e-01,
          -2.9276e-01, -9.3029e-01, -2.5087e-01],
         [ 4.3463e-01,  7.0451e-02, -6.5183e-02, -2.6272e-01,  3.2407e-01,
          -1.0141e-02,  8.9582e-01,  2.6661e-01]],

        [[ 6.8446e-02, -7.2477e-02, -4.0578e-02, -1.5738e-02,  9.3951e-03,
          -1.8731e-01, -1.4599e-02, -3.0612e-01],
         [ 1.9311e-01, -3.7872e-01, -1.6169

In [6]:
# =========================================================
# 7. COMPARE OUTPUT SHAPES
# =========================================================

print("\nShape Comparison:")

if manual_output.shape == builtin_output.shape:
    print("Shape MATCHED")
else:
    print("Shape NOT MATCHED")


# =========================================================
# 8. MAXIMUM DIFFERENCE
# =========================================================

difference = torch.max(
    torch.abs(manual_output - builtin_output)
)

print("\nMaximum Difference:")
print(difference.item())


Shape Comparison:
Shape MATCHED

Maximum Difference:
0.9288848042488098


In [7]:
# =========================================================
# 9. FINAL VALIDATION
# =========================================================

if torch.allclose(
    manual_output,
    builtin_output,
    atol=1e-6
):
    print("\nSUCCESS: Manual BiLSTM matches PyTorch BiLSTM")
else:
    print("\nOutputs are different because the models have different weights.")


Outputs are different because the models have different weights.
